# Historical DAAM walkthrough
Uploaded as `daam.ipynb`. Archived reference; old installation and CUDA-only cells are not the maintained setup. The generator keyword, seed argument, and time-based seed were corrected. Original saved outputs were removed; use the main demo for new runs.


## Setup

In [ ]:
!pip install daam==0.1.0
!pip install accelerate  # this is to reduce CPU model load overhead

In [ ]:
!nvidia-smi

In [ ]:
from diffusers import StableDiffusionPipeline
import daam

In [ ]:
model = StableDiffusionPipeline.from_pretrained('stabilityai/stable-diffusion-2-base')
model = model.to('cuda')

## Global Word Attribution Maps

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

def make_im_subplots(*args):
  fig, ax = plt.subplots(*args)

  for ax_ in ax.flatten():
    ax_.set_xticks([])
    ax_.set_yticks([])

  return fig, ax

prompt = 'An angry, bald man writing papers'

# Trace through generation
with daam.trace(model) as trc:
  output_image = model(prompt, num_inference_steps=20).images[0]
  global_heat_map = trc.compute_global_heat_map()

angry_heat_map = global_heat_map.compute_word_heat_map('angry')
bald_heat_map = global_heat_map.compute_word_heat_map('bald')
papers_heat_map = global_heat_map.compute_word_heat_map('papers')

plt.rcParams['figure.figsize'] = (8, 8)
fig, ax = make_im_subplots(2, 2)

# Original image
ax[0, 0].imshow(output_image)

# Angry heat map
angry_heat_map.plot_overlay(output_image, ax=ax[0, 1])

# Bald heat map
bald_heat_map.plot_overlay(output_image, ax=ax[1, 0])

# Papers heat map
papers_heat_map.plot_overlay(output_image, ax=ax[1, 1])

plt.show()

## Per-Head Attribution Maps

In [ ]:
prompt = 'A cute cat pouncing on a ball'
fig, ax = make_im_subplots(4, 4)

# Trace through generation
with daam.trace(model) as trc:
  output_image = model(prompt, num_inference_steps=20).images[0]

  for head_idx in range(16):
    ax_ = ax[head_idx // 4, head_idx % 4]
    global_heat_map = trc.compute_global_heat_map(head_idx=head_idx, layer_idx=2)
    angry_heat_map = global_heat_map.compute_word_heat_map('cat')
    angry_heat_map.plot_overlay(output_image, ax=ax_)
    ax_.set_title(f'Head {head_idx}')

plt.show()

## Adjectival Entanglement

In [ ]:
# Trace through generation
import time
import torch
import random

def set_seed(seed):
  gen = torch.Generator(device='cuda')
  random.seed(seed)
  np.random.seed(seed)
  torch.cuda.manual_seed(seed)
  torch.manual_seed(seed)

  return gen.manual_seed(seed)

s = 42  # Fixed seed for repeatable comparisons.
gen = set_seed(s)

with daam.trace(model, save_heads=True) as trc:
  blue_image = model('a blue car driving down the street', num_inference_steps=20, generator=gen).images[0]
  blue_map = trc.compute_global_heat_map().compute_word_heat_map('blue')

gen = set_seed(s)

with daam.trace(model, load_heads=True) as trc:
  green_image = model('a green car driving down the street', num_inference_steps=20, generator=gen).images[0]
  green_map = trc.compute_global_heat_map().compute_word_heat_map('green')

gen = set_seed(s)

with daam.trace(model, load_heads=True) as trc:
  red_image = model('a red car driving down the street', num_inference_steps=20, generator=gen).images[0]
  red_map = trc.compute_global_heat_map().compute_word_heat_map('red')

plt.rcParams['figure.figsize'] = (8, 8)
fig, ax = make_im_subplots(2, 2)

# Original images
ax[0, 1].imshow(blue_image)
ax[1, 0].imshow(green_image)
ax[1, 1].imshow(red_image)

# Heat map
blue_map.plot_overlay(blue_image, ax=ax[0, 0])

plt.show()

## Cohyponym Entanglement

In [ ]:
plt.rcParams['figure.figsize'] = (8, 8)
fig, ax = make_im_subplots(2, 3)

with daam.trace(model, save_heads=True) as trc:
  im1 = model('a zebra and a giraffe', num_inference_steps=20).images[0]
  heat_map = trc.compute_global_heat_map()
  zebra_map = heat_map.compute_word_heat_map('zebra')
  giraffe_map = heat_map.compute_word_heat_map('giraffe')

with daam.trace(model, save_heads=True) as trc:
  im2 = model('a crab and a lobster', num_inference_steps=20).images[0]
  heat_map = trc.compute_global_heat_map()
  crab_map = heat_map.compute_word_heat_map('crab')
  lobster_map = heat_map.compute_word_heat_map('lobster')

ax[0, 0].imshow(im1)
ax[1, 0].imshow(im2)
zebra_map.plot_overlay(im1, ax=ax[0, 1])
giraffe_map.plot_overlay(im1, ax=ax[0, 2])

crab_map.plot_overlay(im2, ax=ax[1, 1])
lobster_map.plot_overlay(im2, ax=ax[1, 2])

plt.show()